<img src="../../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# EU government bond yields - data exploration exercise - Solution
---

In this exercise we'll use a dataset of long term government bond yields from [data.europa.eu](https://data.europa.eu/euodp/en/data/dataset/JmCLAeHrvXI80AhjeV3zYQ) to practise exploring data with `pandas`.

### 1. Load in the data

Open the dataset `eu-govt-bonds.tsv` in the `data` folder with a text editing tool like Notepad to examine its structure. This will help determine what options to use when loading the data in. Things to consider:

- is there a header row?
- what is the separator used to separate the different items of data?

Examine the top 5 rows once you've loaded the data in

In [ ]:
import pandas as pd

df = pd.read_csv("../data/eu-govt-bonds.tsv", sep="\t")
df.head()

### 2. Fix the column names

Try to select just the column for January 2020. Do you get an error? If so, why?

Print the column names to see what the problem is, and fix it.

*Hint: when you view a table in Jupyter, you won't necessarily see extra whitespace!*

In [ ]:
"2020M01" in df.columns

Strange, it certainly looks like the column should be called "2020M01"...

In [ ]:
print(df.columns)

In [ ]:
# you can use a list comprehension to trim whitespace for all column names
df.columns = [c.strip() for c in df.columns]
df["2020M01"].head()

### 3. Extract the country code into its own column

The first column of data has the two-letter country code at the end of each value. Extract this into a new column called "country_code", and drop the original column.

In [ ]:
df["country_code"] = df["int_rt,geo\\time"].str[-2:]
df.drop("int_rt,geo\\time", axis=1, inplace=True)
df.head()

### 4. Check (and fix) the data types

Print the data types of the dataset. Do you notice anything strange?

One of the countries is causing all the columns to register as non-numeric. Find the country with invalid values and remove it from the dataset.

Once you've removed it, convert the numeric columns to be the correct numeric type.

In [ ]:
df.dtypes

They should be "float" not "object" - the latter suggests there are string values somewhere...

One trick when suspecting there are non-numeric values in a numeric column is to sort the column and examine the start and end.

In [ ]:
df["2019M02"].sort_values().head()

In [ ]:
df["2019M02"].sort_values().tail()

There we go, one value that's something like ": z".

We could either cheat and find the culprit in Notepad, or do it in code:

In [ ]:
# extract that last value (so we don't have to guess how many spaces there are in it)
offending_value = df["2019M02"].sort_values().values[-1]
print(offending_value)

In [ ]:
df.loc[df["2019M02"] == offending_value, "country_code"]

So it's Estonia. Let's remove it from the data and convert the columns to floats

In [ ]:
df.drop(df[df["country_code"] == "EE"].index, axis=0, inplace=True)

# use a list comprehension to find the column names that should be floats
numeric_cols = [c for c in df.columns if c.startswith("20")]
for c in numeric_cols:
    # convert to float
    df[c] = df[c].astype(float)
    
# verify
df.dtypes

### 5. Load in a second dataset, of country codes and their matching country names 

There is a dataset in the `data` folder called `country-codes.csv` - load this into a new DataFrame.

***Note: if you encounter an error when loading the dataset, it is due to special characters appearing in the file, namely the "Å" in Åland Islands. To bypass this, look at the `encoding` parameter when reading in your csv. [This article](https://docs.python.org/3/library/codecs.html#standard-encodings) details all the possible encodings you can use - look for something that deals with Western European languages. ***

In [ ]:
df_codes = pd.read_csv("../data/country-codes.csv", encoding="latin-1")
df_codes.head()

### 6. Use the country code to get the country name

Merge the rate dataset with country codes to add the country name to your original DataFrame.

***Note: perhaps not all values in your `country_code` column map to the `country-codes` file, so choose your join type wisely! Hint: look at the `how` parameter in the pandas `merge` function.***

In [ ]:
print(len(df))
df_merged = df.merge(df_codes, left_on=["country_code"], right_on=["Alpha-2 code"], how="left")
print(len(df_merged))
# remove some unnecessary columns
df_merged.drop(["Alpha-2 code", "Alpha-3 code"], axis=1, inplace=True)
df_merged.head()

### 7. Check for missing data

Is there any missing data in any of the columns? Which country codes were you not able to match to a row in the reference file?

In [ ]:
df_merged.isnull().sum()

In [ ]:
df_merged[df_merged["Country"].isnull()]

### 8. Explicitly add the country name for the UK

The country code for the UK is "GB", but in the dataset it was coded as "UK" so the join won't have worked for that record.

Explicitly add the country name for the UK for the row where the `country_code` is `UK`

In [ ]:
df_merged.loc[df_merged["country_code"] == "UK", "Country"] = "United Kingdom"

### 9. Find the country with the highest and lowest rates in January 2020

In [ ]:
# Option 1: sort and get head() and tail()
jan_rates = df_merged[["Country", "2020M01"]].sort_values("2020M01")
print(jan_rates.head(1))
print(jan_rates.tail(1))

In [ ]:
# Option 2
# another option is to make the country name the index (the unique/primary key)
# and use .idxmax to find "the index of the highest value" (and idxmin for the minimum)
# we can also combine the two by using .agg and passing a list of aggregations we want to perform together
jan_rates.set_index("Country").agg(["idxmin", "idxmax"])

### 10. Find the country with the smallest *absolute* difference in rate between the first and last months of the dataset

In [ ]:
df_merged["diff"] = abs(df_merged["2020M01"] - df_merged["2019M02"])
df_merged.sort_values("diff").head(1)

In [ ]:
df_merged.drop("diff", axis=1, inplace=True)

### 11. Export your joined/cleaned dataset for use in later exercises

You can use `index=False` for the output to not contain the index column

`df_merged.to_csv("path/to/csv", index=False)`

In [ ]:
df_merged.to_csv("../data/eu-govt-bonds-cleaned.csv", index=False)

### 12. [Advanced] Calculate the mean and median rates per country

This is trickier than it looks because you want the mean and median rates, but rates are in different columns.

One approach is to make the country the index (primary/unique key) and calculate the mean/median of the numeric columns only (so ignoring country code for example). Calling `.mean()` on an entire dataset will calculate the mean *per column* whereas you want the mean *per row*. There is a parameter within `.mean()` that lets you change the direction of the aggregation...

Bonus: do this in the same command, outputting a table with a row per country and a column for the mean and another for the median. *Hint: there's a built in `pandas` function that lets you combine aggregations*

In [ ]:
# Set the index to country AND country code (to allow for codes that didn't match to a country)
# and calculate the mean, but along axis=1 meaning per row, not per column
df_merged.set_index(["country_code", "Country"]).mean(axis=1)

We can do the same as above but with `.median` OR we could combine the two using `.agg`

In [ ]:
# agg takes a list of valid aggregations
df_merged.set_index(["country_code", "Country"]).agg(["mean", "median"], axis=1)

### 13. [Advanced] To make calculations easier, convert the data to a *narrow* format

The idea is rather than having a column per month, have a "month" column and have all the values stored in it. The data should therefore look like this:

| Country | Month   | Rate |
|---------|---------|------|
| Austria | 2019M02 | 0.45 |
| Austria | 2019M03 | 0.38 |

Use [this pandas article](https://pandas.pydata.org/pandas-docs/stable/user_guide/reshaping.html) to help you find the right command.

You can read more about wide vs. narrow data here: [https://en.wikipedia.org/wiki/Wide_and_narrow_data](https://en.wikipedia.org/wiki/Wide_and_narrow_data)

In [ ]:
# Python trick: put brackets around your entire expression and you can then lay it out over multiple lines
df_stacked = (
    df_merged
     .set_index(["country_code", "Country"])
     .stack() # this puts all the month columns in one column
     .reset_index() # convert the resulting Series to a DataFrame
     .rename(columns={"level_2": "month", 0: "rate"}) # rename the resulting default column names
)

df_stacked.head()

### 14. Calculate the mean and median rate for each country again

Hopefully you'll agree it's more natural to do it with a narrow dataset rather than a wide one - just one difference between the two formats!

In [ ]:
df_stacked.groupby("Country")["rate"].agg(["mean", "median"])